# Phase 4a — train the full-collision G1 to follow velocity commands

Isaac Sim 5.1 + Isaac Lab 2.3.2 on a Colab GPU. Runs the repo's own scripts unmodified, so a
result here means the same thing a result on the dev box would.

**What this is for.** The Isaac policy does not walk yet. Two full-size runs are in
`notes/experiments.md` and neither translates — the second one, at upstream's own 4096 envs
and 1792 iterations, learned to *turn on the spot* and never took a step. The diagnosed cause
is that upstream's `feet_air_time` reward is gated on the **linear** command channels only, so
a commanded turn is worth nothing for stepping; `dome_g1/mdp.py::feet_air_time_joystick`
widens that gate and is **committed but never validated**. This notebook validates it, against
two other configurations, with enough compute that "undertrained" is not an available excuse.

| | Run | What it tests |
|---|---|---|
| **A** | `--variant dr` | the gate fix — the hypothesis on the table |
| **C** | `--variant heading` | upstream's task definition on our randomized physics. A **positive control**: if this does not walk, the harness is at fault and A and B are uninterpretable |
| **B** | `--variant dr --reward-scale action_rate_l2=-0.001` | the named fallback lever. `action_rate_l2` saturated at −0.41 and was the dominant term; lowering it trades gait smoothness for locomotion |

Each run is watchdogged: if `feet_air_time` is still flat at iteration 500 it aborts itself,
because a run with that shape provably does not recover (`notes/decisions.md`, 2026-08-08).
That turns a dead 3-hour run into a 25-minute one.

> ### ⚠️ Pick **L4**, not A100 — Runtime → Change runtime type
> NVIDIA lists GPUs without RT cores (**A100, H100**) as unsupported for Isaac Sim 5.1.
> Headless physics may still run, but `--video` brings up the offscreen RTX renderer, which is
> exactly the part that needs them — and video is the deliverable here. Colab has also been
> seen substituting L4 for a requested A100, so cell 1 checks what you actually got rather
> than what you asked for.

> ### Honest status of this notebook
> Nobody has published a working Isaac Sim **5.1** install on Colab; the one documented recipe
> targets 4.5. The GPU plumbing below follows it, and cell 1 is a hard gate rather than a hope.
> If the driver or the Vulkan check fails, the fallback is a rented L40S/A10, where
> `sims/isaac/setup_isaac_cloud.sh` runs unchanged — see `sims/isaac/README.md`.

## 1 — Preflight

Fails loudly and early rather than 30 minutes into a 25 GB install.

In [ ]:
#@title Runtime preflight — GPU, driver, disk, RAM
import os, re, shutil, subprocess, sys

# Isaac Sim 5.1's stated Linux minimum. Colab's driver cannot be upgraded from inside the
# runtime, so this is a property of the machine you were given, not something to work around.
MIN_DRIVER = "580.65.06"
MIN_FREE_GB = 40          # the pip install unpacks ~25 GB; leave room for checkpoints
OVERRIDE_DRIVER_CHECK = False   # set True to try anyway — the minimum is stated, not proven

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

def vtuple(v):
    return tuple(int(x) for x in re.findall(r"\d+", v))

problems, warnings = [], []

smi = sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
if not smi:
    problems.append("No GPU. Runtime -> Change runtime type -> GPU (L4).")
    gpu = driver = ""
else:
    gpu, vram, driver = [f.strip() for f in smi.split(",")]
    print(f"GPU     {gpu}")
    print(f"VRAM    {vram}")
    print(f"driver  {driver}")

    if vtuple(driver) < vtuple(MIN_DRIVER):
        msg = f"driver {driver} < Isaac Sim 5.1's minimum {MIN_DRIVER}"
        (warnings if OVERRIDE_DRIVER_CHECK else problems).append(msg)
    if re.search(r"A100|H100", gpu):
        problems.append(
            f"{gpu} has no RT cores, which NVIDIA lists as unsupported for Isaac Sim 5.1. "
            "Headless training may still work but --video will not, and video is the point. "
            "Switch to L4.")
    # 4096 full-collision G1 envs measured at 5.05 GB on the dev box (notes/setup.md).
    if int(re.findall(r"\d+", vram)[0]) < 16000:
        warnings.append(f"{vram} is below Isaac's 16 GB minimum; 4096 envs needs ~5 GB, "
                        "but the renderer for --video wants several more.")

free_gb = shutil.disk_usage("/content").free / 2**30
print(f"disk    {free_gb:.0f} GB free on /content")
if free_gb < MIN_FREE_GB:
    problems.append(f"only {free_gb:.0f} GB free on /content; need >= {MIN_FREE_GB} GB.")

ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
print(f"RAM     {ram_gb:.0f} GB")
if ram_gb < 20:
    warnings.append(f"{ram_gb:.0f} GB RAM. Building a 4096-env scene is memory-hungry; "
                    "if it is OOM-killed, pick a High-RAM runtime.")

print()
for w in warnings:
    print("WARNING:", w)
if problems:
    print()
    for p in problems:
        print("BLOCKED:", p)
    print("\nFallback: a rented L40S / A10 box, where sims/isaac/setup_isaac_cloud.sh runs "
          "unchanged. See sims/isaac/README.md.")
    raise SystemExit("preflight failed — see above")
print("\npreflight OK")

## 2 — Configuration and Drive

`runs/` is symlinked onto Drive so a disconnect costs at most one `save_interval` (25
iterations). This repo lost a 4-hour sweep to a reclaimed VM once already; the LingBot
notebook gained the same symlink for the same reason.

In [ ]:
#@title Settings
REPO_URL   = "https://github.com/adikothuri3/geologic_dome_sim.git"
GIT_REF    = "isaac-pivot"      #@param {type:"string"}
PERSIST_DRIVE = True            #@param {type:"boolean"}

# If the repo is private: Colab -> 🔑 Secrets -> add GH_TOKEN (a PAT with `repo` scope).
GH_TOKEN = ""
try:
    from google.colab import userdata
    GH_TOKEN = userdata.get("GH_TOKEN")
except Exception:
    pass

import pathlib
REPO       = pathlib.Path("/content/GeologicDome")
VENV       = pathlib.Path("/content/venvs/isaac")
PY         = str(VENV / "bin" / "python")
LAB_DIR    = "/content/src/IsaacLab"
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/GeologicDome")

if PERSIST_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    for sub in ("isaac_runs", "cache", "results"):
        (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
    print("Drive ready at", DRIVE_ROOT)
else:
    print("PERSIST_DRIVE off — a disconnect loses every checkpoint. You have been told.")

In [ ]:
#@title Clone the repo at a pinned ref
import os, subprocess

url = REPO_URL
if GH_TOKEN:
    url = REPO_URL.replace("https://", f"https://{GH_TOKEN}@")

if not REPO.exists():
    subprocess.run(["git", "clone", url, str(REPO)], check=True)
subprocess.run(["git", "fetch", "--all", "--tags", "--quiet"], cwd=REPO, check=True)
# Re-running this cell after an interrupted session can find experiments.md dirty from a
# run whose row was written but not yet harvested. The rows accumulate on Drive, not here.
subprocess.run(["git", "checkout", "--", "notes/experiments.md"], cwd=REPO, check=False)
subprocess.run(["git", "checkout", GIT_REF], cwd=REPO, check=True)
subprocess.run(["git", "pull", "--ff-only", "--quiet"], cwd=REPO, check=False)

COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO,
                        capture_output=True, text=True, check=True).stdout.strip()
print(f"{GIT_REF} @ {COMMIT}")

# runs/ is gitignored, so pointing it at Drive does not dirty the tree -- which matters,
# because the trainer refuses to start on a dirty tree (the experiments.md row has to name
# reproducible code).
runs_isaac = REPO / "runs" / "isaac"
if PERSIST_DRIVE:
    (REPO / "runs").mkdir(exist_ok=True)
    if not runs_isaac.is_symlink():
        if runs_isaac.exists():
            import shutil as _sh; _sh.rmtree(runs_isaac)
        runs_isaac.symlink_to(DRIVE_ROOT / "isaac_runs")
else:
    runs_isaac.mkdir(parents=True, exist_ok=True)
RUNS = runs_isaac
print("runs/isaac ->", os.path.realpath(RUNS))

# Used by every subprocess from here on.
ENV = {**os.environ, "OMNI_KIT_ACCEPT_EULA": "YES", "TMPDIR": "/content/tmp"}
KIT_QUIET = ["--/log/level=error", "--/log/fileLogLevel=error",
             "--/log/outputStreamLevel=error"]

## 3 — Install

Two scripts, both in the repo. The first is Colab-specific (Python 3.11, the Vulkan/EGL
manifests Colab omits, scratch space); the second is the same Linux installer a rented box
would use, carrying the three pins that each cost real debugging time.

**This is the slow cell: ~25 GB and 20–40 minutes.** It is idempotent, so a re-run after a
disconnect skips what is already there.

In [ ]:
#@title Colab GPU plumbing (python3.11, Vulkan ICD/EGL manifests, TMPDIR)
import os, subprocess
os.environ["DRIVE_CACHE"] = str(DRIVE_ROOT / "cache") if PERSIST_DRIVE else ""
rc = subprocess.run(["bash", "sims/isaac/setup_colab_gpu.sh"], cwd=REPO).returncode
if rc:
    raise SystemExit(
        "setup_colab_gpu.sh failed. If it was the Vulkan gate, Kit will not start a "
        "renderer on this runtime and --video is out; see the hints it printed.")

In [ ]:
#@title Isaac Sim 5.1.0 + Isaac Lab 2.3.2  (~25 GB, 20-40 min)
import os, subprocess
env = {**os.environ,
       "VENV_DIR": str(VENV), "LAB_DIR": LAB_DIR,
       "TMPDIR": "/content/tmp", "OMNI_KIT_ACCEPT_EULA": "YES"}
rc = subprocess.run(["bash", "sims/isaac/setup_isaac_cloud.sh"], cwd=REPO, env=env).returncode
if rc:
    raise SystemExit("setup_isaac_cloud.sh failed — read the last step it printed.")

## 4 — Gates

Three cheap checks, in order, before spending hours of GPU time.

**Gate c is the one that earns its keep.** Upstream's G1 config *disables* almost all of
Isaac's own randomization — `push_robot = None`, `add_base_mass = None`, friction ranges
collapsed to point values — so a task that randomizes nothing trains, logs and plots exactly
like one that does. Gate c reads the per-environment spread back out of PhysX instead of
trusting the config.

In [ ]:
#@title Gates a / b / c
import subprocess, time

def gate(letter, *extra):
    print(f"\n{'='*70}\nGATE {letter}\n{'='*70}", flush=True)
    t = time.time()
    rc = subprocess.run([PY, "sims/isaac/scripts/check_isaac.py", "--gate", letter,
                         *extra, *KIT_QUIET], cwd=REPO, env=ENV).returncode
    print(f"gate {letter}: {'PASS' if rc == 0 else 'FAIL'}  ({time.time()-t:.0f}s)", flush=True)
    return rc == 0

# (a) the app comes up headless at all. First run is slow -- shader cache.
# (b) the full-collision G1 builds and steps, and reports VRAM + throughput at the size we
#     are about to train at, on THIS gpu rather than the dev box's table.
# (c) every domain-randomization term actually reaches PhysX.
ok = gate("a") and gate("b", "--num_envs", "4096") and gate("c", "--num_envs", "32")
if not ok:
    raise SystemExit("a gate failed — do not start a training run on this")
print("\nall gates green")

## 5 — Training

The driver below runs each configuration as a subprocess, retrying with `--resume` if the
process dies — so a Colab hiccup costs at most 25 iterations.

Two things it deliberately does **not** do:

- **It does not print RSL-RL's per-iteration table.** 3000 iterations of it would be ~120,000
  lines. Full output goes to a log file; what is printed here is the reward-term digest the
  trainer's guards write to `progress.jsonl`.
- **It does not retry a collapsed run.** Resuming a policy whose stepping reward provably
  will not lift is the one thing the loop must never do.

It decides which of those happened by reading `outcome.json`, **not** the exit code. Measured
2026-08-08: `raise SystemExit(3)` after `simulation_app.close()` yields an exit code of `0`,
because Kit owns process shutdown — the same reason `sys.exit("message")` raises `TypeError`
inside a running app. A missing `outcome.json` means the process never reached its own
teardown, which is exactly the case worth retrying.

**Read `feet_air_time`, not `reward`.** In the failed 4096-env run mean reward climbed −30 →
+4.11 across the whole run, entirely on the yaw term, while the robot never took a step.
Healthy is `feet_air_time` past ~0.05 by iteration 300–500 and still rising, with `track_lin`
above 0.5 (standing still scores **0.37**).

In [ ]:
#@title The run driver
import json, os, pathlib, subprocess, sys, time

NUM_ENVS       = 4096   #@param {type:"integer"}
MAX_ITERATIONS = 3000   #@param {type:"integer"}
ABORT_IF_FLAT  = 500    #@param {type:"integer"}
MAX_ATTEMPTS   = 6
PRINT_EVERY    = 25

LOGS = pathlib.Path("/content/logs"); LOGS.mkdir(exist_ok=True)
ROWS = (DRIVE_ROOT / "results" / "experiments_rows.md") if PERSIST_DRIVE \
       else pathlib.Path("/content/experiments_rows.md")
ROWS.parent.mkdir(parents=True, exist_ok=True)


def _f(v, spec="{:.4f}"):
    return spec.format(v) if isinstance(v, (int, float)) else "  --  "


def find_run_dir(run_name):
    """The trainer names its directory <utc-date>-isaac-<run_name>; a run that spans
    midnight UTC would otherwise be looked for under the wrong date."""
    hits = sorted(RUNS.glob(f"*-isaac-{run_name}"))
    return hits[-1] if hits else None


def highest_iter(run_dir):
    if run_dir is None or not run_dir.exists():
        return -1
    ck = [int(p.stem.split("_")[1]) for p in run_dir.glob("model_*.pt")
          if p.stem.split("_")[1].isdigit()]
    return max(ck) if ck else -1


def harvest_rows(label):
    """Take the experiments.md rows the trainer just appended, then restore the file.

    The trainer's clean-tree gate refuses to start on a dirty tree -- correctly, since the
    row has to name reproducible code -- so leaving run A's row in place would block run C.
    The rows are not discarded: they accumulate in ROWS (on Drive) to be applied to the
    real repo afterwards, which is where they belong anyway.
    """
    diff = subprocess.run(["git", "diff", "--unified=0", "--", "notes/experiments.md"],
                          cwd=REPO, capture_output=True, text=True).stdout
    added = [l[1:] for l in diff.splitlines()
             if l.startswith("+") and not l.startswith("+++") and l[1:].strip()]
    if added:
        with ROWS.open("a", encoding="utf-8") as fh:
            fh.write(f"\n<!-- {label} -->\n" + "\n".join(added) + "\n")
    subprocess.run(["git", "checkout", "--", "notes/experiments.md"], cwd=REPO, check=False)
    return added


def drain_progress(run_dir, seen):
    """Print new progress.jsonl records. Returns the new cursor.

    Tolerates a partially-written last line: the trainer appends while we read, so a failed
    parse means 'not flushed yet', not 'corrupt' -- retry on the next poll without advancing.
    """
    path = run_dir / "progress.jsonl"
    if not path.exists():
        return seen
    lines = path.read_text(encoding="utf-8", errors="ignore").splitlines()
    while seen < len(lines):
        try:
            rec = json.loads(lines[seen])
        except json.JSONDecodeError:
            break
        seen += 1
        it = rec.get("iteration", 0)
        if it % PRINT_EVERY and it != 0:
            continue
        flag = "step" if rec.get("gated") else "    "
        print(f"  it {it:>5} {flag}  air {_f(rec.get('feet_air_time'))}"
              f"  lin {_f(rec.get('track_lin_vel_xy_exp'))}"
              f"  ang {_f(rec.get('track_ang_vel_z_exp'))}"
              f"  rew {_f(rec.get('mean_reward'), '{:>7.2f}')}"
              f"  lr {_f(rec.get('learning_rate'), '{:.1e}')}", flush=True)
    return seen


def train(label, run_name, extra_args=(), max_iterations=None):
    max_iterations = max_iterations or MAX_ITERATIONS
    print(f"\n{'='*78}\n{label}\n{'='*78}", flush=True)

    run_dir = find_run_dir(run_name)
    if highest_iter(run_dir) >= max_iterations - 1:
        print(f"already at iteration {highest_iter(run_dir)} — nothing to do")
        return {"label": label, "run_dir": run_dir, "status": "ok"}

    seen, status = 0, "failed"
    for attempt in range(1, MAX_ATTEMPTS + 1):
        run_dir = find_run_dir(run_name)
        argv = [PY, "sims/isaac/scripts/train_g1_flat.py",
                "--num_envs", str(NUM_ENVS),
                "--max_iterations", str(max_iterations),
                "--abort-if-flat", str(ABORT_IF_FLAT),
                "--run_name", run_name, *extra_args, *KIT_QUIET]
        if highest_iter(run_dir) >= 0:
            argv += ["--resume", str(run_dir)]
            print(f"attempt {attempt}: resuming from iteration {highest_iter(run_dir)}",
                  flush=True)
        elif attempt > 1:
            print(f"attempt {attempt}: restarting from scratch (no checkpoint yet)",
                  flush=True)

        log_path = LOGS / f"{run_name}-attempt{attempt}.log"
        with log_path.open("w", encoding="utf-8") as log:
            proc = subprocess.Popen(argv, cwd=REPO, env=ENV, stdout=log,
                                    stderr=subprocess.STDOUT, text=True)
            while proc.poll() is None:
                rd = find_run_dir(run_name)
                if rd is not None:
                    seen = drain_progress(rd, seen)
                time.sleep(5)
        run_dir = find_run_dir(run_name)
        if run_dir is not None:
            seen = drain_progress(run_dir, seen)

        rows = harvest_rows(f"{label} attempt {attempt}")
        for r in rows:
            print("  row:", r[:160], flush=True)

        # outcome.json, not proc.returncode -- Kit forces the exit code to 0 on shutdown.
        # Absent means the process never reached its own teardown: a crash or a disconnect.
        outcome = {}
        if run_dir is not None and (run_dir / "outcome.json").exists():
            outcome = json.loads((run_dir / "outcome.json").read_text())
        verdict = outcome.get("status")

        if verdict == "ok":
            status = "ok"; break
        if verdict == "collapsed":
            # The watchdog fired. A result, not a crash -- and resuming it would only buy
            # more of the same (notes/decisions.md, 2026-08-08).
            status = "collapsed"
            print("  " + outcome.get("metrics", "")[:400], flush=True)
            break
        why = f"outcome {verdict}" if verdict else f"no outcome.json (exit {proc.returncode})"
        print(f"  {why} — last 25 lines of {log_path.name}:", flush=True)
        print("\n".join(log_path.read_text(errors="ignore").splitlines()[-25:]), flush=True)
        if run_dir is None:
            # The trainer bailed before it even made its directory, so it never reached the
            # SimulationApp — a dirty tree, a bad argument, a missing venv. None of those
            # get better by being retried five more times.
            print("  no run directory was created: the trainer exited before starting. "
                  "Fix the cause above rather than retrying.", flush=True)
            break
        if verdict == "FAILED":
            # The trainer caught a real exception and logged it. Retrying resumes from the
            # last checkpoint, which is worth one or two goes (transient CUDA faults are a
            # thing) but not six.
            if attempt >= 2:
                print("  failing twice on a caught exception is not transient; stopping",
                      flush=True)
                break
        if attempt == MAX_ATTEMPTS:
            print(f"  giving up after {MAX_ATTEMPTS} attempts", flush=True)

    print(f"\n{label}: {status.upper()}  ({run_dir})", flush=True)
    return {"label": label, "run_dir": run_dir, "status": status}


COMPLETED = []
print("driver ready — run the three cells below in order")

In [ ]:
#@title RUN A — the gate fix  (`feet_air_time_joystick`)
COMPLETED.append(train("RUN A — gate fix", "g1fc-flat-dr"))

In [ ]:
#@title RUN C — the positive control  (upstream's task definition)
COMPLETED.append(train("RUN C — heading control", "g1fc-flat-heading",
                       extra_args=["--variant", "heading"]))

In [ ]:
#@title RUN B — the fallback lever  (action_rate_l2 -0.005 -> -0.001)
COMPLETED.append(train("RUN B — action_rate lever", "g1fc-flat-dr-actionrate",
                       extra_args=["--reward-scale", "action_rate_l2=-0.001"]))

## 6 — Score and render

`play_g1_flat.py` holds one velocity command for a whole 10 s clip and reports the error
directly, over a five-command sweep. A reward curve says a policy improved; this says whether
the robot realises the velocity it was asked for.

**How to read it.** MAE *well under* the command magnitude on every channel is a working
policy. MAE **equal to** the command magnitude is the signature of zero motion — it is what
both previous runs scored, at 100 % survival, because a policy that stands still tracks a
zero command perfectly and every other command not at all.

`--checkpoint best` loads the best iteration by `walk_score`, not the last one. On the MuJoCo
track that distinction was worth 4.6 reward: both runs peaked near 80M steps and then drifted
down, so the saved artefact got worse while the log looked like it was flattening out.

In [ ]:
#@title Score every finished run, render one clip per command
import subprocess

def score(entry):
    run_dir = entry["run_dir"]
    if run_dir is None or not any(run_dir.glob("model_*.pt")):
        print(f"{entry['label']}: no checkpoints, skipping"); return
    print(f"\n{'='*78}\n{entry['label']}  ({run_dir.name})\n{'='*78}", flush=True)

    base = [PY, "sims/isaac/scripts/play_g1_flat.py", str(run_dir), "--video", *KIT_QUIET]
    # A run that aborted before the guards' 100-iteration warmup has no best.json; fall
    # back to the final weights rather than reporting nothing for it.
    rc = subprocess.run(base + ["--checkpoint", "best"], cwd=REPO, env=ENV).returncode
    if rc:
        print("  no best.json — scoring the final checkpoint instead", flush=True)
        subprocess.run(base, cwd=REPO, env=ENV, check=False)
    # --out into the run directory, NOT plot_play.py's default of reports/. reports/ is
    # tracked, so a plot written there leaves the tree dirty and the next training cell
    # would be refused by the trainer's clean-tree gate. This also keeps the plot with the
    # run it describes, which is what gets zipped.
    subprocess.run([PY, "sims/isaac/scripts/plot_play.py", str(run_dir),
                    "--out", str(run_dir / "tracking.png")], cwd=REPO, env=ENV, check=False)

for entry in COMPLETED:
    score(entry)

In [ ]:
#@title The videos
import base64, json
from IPython.display import HTML, display

def show(entry):
    run_dir = entry["run_dir"]
    vdir = (run_dir / "videos") if run_dir else None
    if vdir is None or not vdir.exists():
        print(f"{entry['label']}: no videos"); return
    if (vdir / "index.json").exists():
        blob = json.loads((vdir / "index.json").read_text())
    else:
        # index.json is written after env.close(), which Kit can take down with it. The
        # clips are on disk either way, so fall back to whatever is there rather than
        # reporting no video for a run that rendered fine.
        clips = sorted(vdir.glob("*.mp4"))
        blob = {"checkpoint": "?", "clips": [{"command": p.stem, "video": p.name}
                                             for p in clips]}
        print(f"  (no index.json; showing {len(clips)} clip(s) by filename)")

    best = run_dir / "best.json"
    caption = entry["label"]
    if best.exists():
        b = json.loads(best.read_text())
        walked = "stepping" if b.get("gated") else "NO STEPPING — not a walking policy"
        caption += (f" — checkpoint {blob['checkpoint']}, iteration {b.get('best_iteration')}"
                    f", walk_score {b.get('best_walk_score')} ({walked})")
    display(HTML(f"<h3>{caption}</h3>"))

    for clip in blob["clips"]:
        path = run_dir / "videos" / clip["video"]
        if not path.exists():
            continue
        b64 = base64.b64encode(path.read_bytes()).decode()
        display(HTML(
            f"<div style='display:inline-block;margin:6px;text-align:center'>"
            f"<div style='font:13px monospace'>{clip['command']}</div>"
            # A data URI, because Colab's iframe cannot fetch a local file path.
            f"<video width=380 controls loop src='data:video/mp4;base64,{b64}'></video></div>"))

    png = run_dir / "tracking.png"
    if png.exists():
        display(HTML(f"<img src='data:image/png;base64,"
                     f"{base64.b64encode(png.read_bytes()).decode()}' width=900>"))

for entry in COMPLETED:
    show(entry)

## 7 — Bring it home

Every run needs a row in `notes/experiments.md` — the trainer wrote them, and the driver
harvested them here so the tree stayed clean for the next run. Paste them into the real repo
(**rows are never deleted**, including the ones recording a run that did not walk: failed runs
with honest takeaways are the point of that file).

In [ ]:
#@title The experiments.md rows, and the summary table
import json

print(ROWS.read_text(encoding="utf-8") if ROWS.exists() else "(no rows harvested)")

print("\n" + "="*90)
print(f"{'run':<28} {'status':<11} {'walked':<7} {'best it':>8} {'walk_score':>11}")
print("-"*90)
for e in COMPLETED:
    rd, b = e["run_dir"], {}
    if rd and (rd / "best.json").exists():
        b = json.loads((rd / "best.json").read_text())
    print(f"{e['label'][:27]:<28} {e['status']:<11} "
          f"{str(bool(b.get('gated'))):<7} {str(b.get('best_iteration')):>8} "
          f"{str(b.get('best_walk_score')):>11}")

print("\nA run whose `walked` is False did not lift the stepping reward. Its play_metrics.json "
      "will show MAE equal to the command magnitude on every channel — that is what a policy "
      "outputting zero velocity scores, not a broken evaluation.")

In [ ]:
#@title Zip the runs for download
import shutil, pathlib
from google.colab import files

OUT = pathlib.Path("/content/isaac_phase4a_results")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir()

for e in COMPLETED:
    rd = e["run_dir"]
    if rd is None or not rd.exists():
        continue
    dst = OUT / rd.name
    dst.mkdir()
    # Everything except the intermediate checkpoints: 3000 iterations at save_interval 25
    # is 120 files of ~10 MB, and the two that matter are model_best.pt and the last one.
    keep = ["config.json", "progress.jsonl", "best.json", "outcome.json", "model_best.pt",
            "play_metrics.json", "play_timeseries.json", "tracking.png"]
    for name in keep:
        if (rd / name).exists():
            shutil.copy2(rd / name, dst / name)
    latest = sorted(rd.glob("model_*.pt"),
                    key=lambda p: int(p.stem.split("_")[1]) if p.stem.split("_")[1].isdigit() else -1)
    if latest:
        shutil.copy2(latest[-1], dst / latest[-1].name)
    if (rd / "videos").exists():
        shutil.copytree(rd / "videos", dst / "videos")

if ROWS.exists():
    shutil.copy2(ROWS, OUT / "experiments_rows.md")

archive = shutil.make_archive("/content/isaac_phase4a_results", "zip", OUT)
print(archive, f"{pathlib.Path(archive).stat().st_size / 2**20:.1f} MB")
if PERSIST_DRIVE:
    shutil.copy2(archive, DRIVE_ROOT / "results" / pathlib.Path(archive).name)
    print("copied to Drive")
files.download(archive)

In [ ]:
#@title (optional) Save the Kit shader caches to Drive for the next session
import shutil, pathlib, os
if PERSIST_DRIVE:
    cache = DRIVE_ROOT / "cache"
    for name, src in [("ov", pathlib.Path.home() / ".cache/ov"),
                      ("ComputeCache", pathlib.Path.home() / ".nv/ComputeCache")]:
        if src.exists():
            dst = cache / name
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print("saved", name)
    print("next session's setup_colab_gpu.sh will restore these and skip the cold "
          "shader compile (~30 min)")